In [3]:
!pip install wfdb scipy scikit-learn torch --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 101.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


In [4]:
import ast
import os
import zipfile
import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, filtfilt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

In [ ]:
DATA_PATH = "/content/sample_data/data"
SAMPLING_RATE = 100
BATCH_SIZE = 64
EPOCHS = 30
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_SAVE_PATH = os.path.join(os.path.dirname(DATA_PATH), "ptbxl_cnn_best.pt") # Adjusted path
CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

In [ ]:
print("Using device:", DEVICE)

Using device: cuda


In [23]:
import os
import zipfile
import glob # For more robust file searching

def extract_records_zip():
    """Extract records100.zip from DATA_PATH if it exists and verify content."""
    zip_file_path = os.path.join(DATA_PATH, 'records100.zip')
    records_extracted_dir = os.path.join(DATA_PATH, 'records100')

    # Ensure DATA_PATH exists
    os.makedirs(DATA_PATH, exist_ok=True)

    if os.path.exists(zip_file_path):
        print(f"Found zip file: {zip_file_path}")
        print(f"Extracting to {DATA_PATH}...")

        try:
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(DATA_PATH)
            print(f" Successfully extracted {os.path.basename(zip_file_path)}")

            # --- Start of enhanced verification ---
            print("Verifying extracted content:")
            if os.path.exists(records_extracted_dir):
                print(f"  Directory '{records_extracted_dir}' exists.")
                # Look for subdirectories like '00000', '01000', etc.
                subdirs = [d for d in os.listdir(records_extracted_dir)
                           if os.path.isdir(os.path.join(records_extracted_dir, d)) and d.isdigit()]
                if subdirs:
                    print(f"  Found {len(subdirs)} subdirectories (e.g., '{subdirs[0]}') inside 'records100'.")
                    # Check for .dat/.hea files in the first found subdirectory
                    first_subdir_path = os.path.join(records_extracted_dir, subdirs[0])
                    files_in_subdir = glob.glob(os.path.join(first_subdir_path, '*.dat')) + \
                                      glob.glob(os.path.join(first_subdir_path, '*.hea'))
                    if files_in_subdir:
                        print(f"  Found {len(files_in_subdir)} .dat/.hea files in '{first_subdir_path}'. First 5: {files_in_subdir[:5]}")
                    else:
                        print(f"  Warning: No .dat/.hea files found in first subdirectory '{first_subdir_path}'.")
                else:
                    print(f"  Warning: No numerical subdirectories found in '{records_extracted_dir}'.")
                    # Fallback check: maybe files were extracted directly into records_extracted_dir
                    files_directly_in_records = glob.glob(os.path.join(records_extracted_dir, '*.dat')) + \
                                                glob.glob(os.path.join(records_extracted_dir, '*.hea'))
                    if files_directly_in_records:
                        print(f"  However, found {len(files_directly_in_records)} .dat/.hea files directly in '{records_extracted_dir}'. First 5: {files_directly_in_records[:5]}")
                    else:
                        print(f"  No .dat/.hea files found directly in '{records_extracted_dir}'.")
            else:
                print(f"  Warning: Directory '{records_extracted_dir}' was NOT created after extraction. Checking if files are directly in {DATA_PATH}.")
                files_directly_in_data_path = glob.glob(os.path.join(DATA_PATH, '*.dat')) + \
                                              glob.glob(os.path.join(DATA_PATH, '*.hea'))
                if files_directly_in_data_path:
                    print(f"  Found {len(files_directly_in_data_path)} .dat/.hea files directly in '{DATA_PATH}'. This is unexpected. First 5: {files_directly_in_data_path[:5]}")
                else:
                    print(f"  No .dat/.hea files found directly in '{DATA_PATH}' either. Data extraction might have failed or the zip is empty/corrupt.")
            # --- End of enhanced verification ---

        except zipfile.BadZipFile:
            print(f"Error: The file '{zip_file_path}' is not a valid zip file. It might be corrupted or incomplete.")
        except Exception as e:
            print(f"Error during extraction: {e}")
    else:
        print(f"No zip file found at '{zip_file_path}'. Looking for existing extracted files...")
        if os.path.exists(records_extracted_dir):
            subdirs = [d for d in os.listdir(records_extracted_dir)
                       if os.path.isdir(os.path.join(records_extracted_dir, d)) and d.isdigit()]
            if subdirs:
                print(f"  Found existing '{records_extracted_dir}' with {len(subdirs)} subdirectories.")
            else:
                print(f"  Directory '{records_extracted_dir}' exists but contains no numerical subdirectories.")
        else:
            print(f"  Warning: Neither '{zip_file_path}' nor '{records_extracted_dir}' found. Data is likely missing.")

# Run the extraction before loading data
extract_records_zip()

Found zip file: /content/sample_data/data/records100.zip
Extracting to /content/sample_data/data...
 Successfully extracted records100.zip
Verifying extracted content:
  Directory '/content/sample_data/data/records100' exists.
  Found 22 subdirectories (e.g., '14000') inside 'records100'.
  Found 2000 .dat/.hea files in '/content/sample_data/data/records100/14000'. First 5: ['/content/sample_data/data/records100/14000/14754_lr.dat', '/content/sample_data/data/records100/14000/14881_lr.dat', '/content/sample_data/data/records100/14000/14983_lr.dat', '/content/sample_data/data/records100/14000/14337_lr.dat', '/content/sample_data/data/records100/14000/14014_lr.dat']


In [ ]:
df = pd.read_csv(os.path.join(DATA_PATH, "ptbxl_database.csv"), index_col="ecg_id")
df.scp_codes = df.scp_codes.apply(ast.literal_eval)

agg_df = pd.read_csv(os.path.join(DATA_PATH, "scp_statements.csv"), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = [agg_df.loc[k].diagnostic_class for k in y_dic.keys() if k in agg_df.index]
    return list(set(tmp))

df["diagnostic_superclass"] = df.scp_codes.apply(aggregate_diagnostic)

# Drop records with zero mapped superclasses
df = df[df["diagnostic_superclass"].apply(len) > 0].copy()

def multi_hot(label_list):
    return np.array([1.0 if c in label_list else 0.0 for c in CLASSES], dtype=np.float32)

df["label_vec"] = df["diagnostic_superclass"].apply(multi_hot)

In [ ]:
train_df = df[df.strat_fold.isin(range(1, 9))]
val_df   = df[df.strat_fold == 9]
test_df  = df[df.strat_fold == 10]

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

Train: 17111  Val: 2156  Test: 2163


In [ ]:
def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=100, order=4):
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal, axis=0)

def preprocess_signal(signal):
    # signal shape: (1000, 12)
    signal = bandpass_filter(signal, fs=SAMPLING_RATE)
    mean = signal.mean(axis=0, keepdims=True)
    std = signal.std(axis=0, keepdims=True) + 1e-8
    signal = (signal - mean) / std
    return signal.astype(np.float32)

In [ ]:
class PTBXLDataset(Dataset):
    def __init__(self, dataframe, data_path, sampling_rate):
        self.df = dataframe
        self.data_path = data_path
        self.sampling_rate = sampling_rate

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = row["filename_lr"] if self.sampling_rate == 100 else row["filename_hr"]
        # Use correct path including records100
        record = wfdb.rdrecord(os.path.join(self.data_path, 'records100', filepath))
        signal = record.p_signal  # Shape is (1000, 12)
        signal = preprocess_signal(signal) # Processes but keeps channels last
        # Transpose here once to get (12, 1000) for the Conv1d layers
        signal = torch.tensor(signal.T, dtype=torch.float32)
        label = torch.tensor(row["label_vec"], dtype=torch.float32)
        return signal, label

# Redefine DataLoaders with the updated class
train_ds = PTBXLDataset(train_df, DATA_PATH, SAMPLING_RATE)
val_ds   = PTBXLDataset(val_df, DATA_PATH, SAMPLING_RATE)
test_ds  = PTBXLDataset(test_df, DATA_PATH, SAMPLING_RATE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
class ECGCNN(nn.Module):
    def __init__(self, in_channels=12, n_classes=5):
        super().__init__()
        self.block1 = self._conv_block(in_channels, 32)
        self.block2 = self._conv_block(32, 64)
        self.block3 = self._conv_block(64, 128)
        self.block4 = self._conv_block(128, 256)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, n_classes)
        )

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv1d(in_c, out_c, kernel_size=7, padding=3),
            nn.BatchNorm1d(out_c),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x).squeeze(-1)
        return self.fc(x)  # raw logits, sigmoid applied via loss (BCEWithLogits)

model = ECGCNN(in_channels=12, n_classes=len(CLASSES)).to(DEVICE)

In [ ]:
label_matrix = np.stack(train_df["label_vec"].values)
pos_counts = label_matrix.sum(axis=0)
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-8), dtype=torch.float32).to(DEVICE)
print("Positive class weights (for imbalance):", pos_weight.cpu().numpy())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)


Positive class weights (for imbalance): [1.2493756 2.89861   3.080849  3.3739774 7.067421 ]


In [ ]:
def run_epoch(loader, train_mode=True):
    model.train() if train_mode else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for signals, labels in loader:
            signals, labels = signals.to(DEVICE), labels.to(DEVICE)

            if train_mode:
                optimizer.zero_grad()

            logits = model(signals)
            loss = criterion(logits, labels)

            if train_mode:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * signals.size(0)
            all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    # Per-class AUC (skip classes with no positive samples in this split)
    aucs = []
    for i, cls in enumerate(CLASSES):
        if len(np.unique(all_labels[:, i])) > 1:
            aucs.append(roc_auc_score(all_labels[:, i], all_preds[:, i]))
        else:
            aucs.append(float("nan"))

    return avg_loss, aucs

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_aucs = run_epoch(train_loader, train_mode=True)
    val_loss, val_aucs = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)

    mean_val_auc = np.nanmean(val_aucs)
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Mean AUC: {mean_val_auc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  -> Saved new best model (val_loss={val_loss:.4f})")

Epoch 01 | Train Loss: 0.6736 | Val Loss: 0.6720 | Val Mean AUC: 0.8886
  -> Saved new best model (val_loss=0.6720)
Epoch 02 | Train Loss: 0.5944 | Val Loss: 0.5953 | Val Mean AUC: 0.9017
  -> Saved new best model (val_loss=0.5953)
Epoch 03 | Train Loss: 0.5745 | Val Loss: 0.6170 | Val Mean AUC: 0.9018
Epoch 04 | Train Loss: 0.5580 | Val Loss: 0.5850 | Val Mean AUC: 0.9052
  -> Saved new best model (val_loss=0.5850)
Epoch 05 | Train Loss: 0.5455 | Val Loss: 0.5896 | Val Mean AUC: 0.9054
Epoch 06 | Train Loss: 0.5328 | Val Loss: 0.5997 | Val Mean AUC: 0.9078
Epoch 07 | Train Loss: 0.5225 | Val Loss: 0.5874 | Val Mean AUC: 0.9061
Epoch 08 | Train Loss: 0.5138 | Val Loss: 0.5883 | Val Mean AUC: 0.9053
Epoch 09 | Train Loss: 0.4831 | Val Loss: 0.6137 | Val Mean AUC: 0.9097
Epoch 10 | Train Loss: 0.4681 | Val Loss: 0.5644 | Val Mean AUC: 0.9109
  -> Saved new best model (val_loss=0.5644)
Epoch 11 | Train Loss: 0.4642 | Val Loss: 0.5955 | Val Mean AUC: 0.9122
Epoch 12 | Train Loss: 0.4490 | 

In [ ]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
test_loss, test_aucs = run_epoch(test_loader, train_mode=False)

print("\n=== Final Test Results ===")
print(f"Test Loss: {test_loss:.4f}")
for cls, auc in zip(CLASSES, test_aucs):
    print(f"  {cls}: AUC = {auc:.4f}")
print(f"Mean Test AUC: {np.nanmean(test_aucs):.4f}")


=== Final Test Results ===
Test Loss: 0.5587
  NORM: AUC = 0.9413
  MI: AUC = 0.9252
  STTC: AUC = 0.9350
  CD: AUC = 0.9230
  HYP: AUC = 0.8395
Mean Test AUC: 0.9128


In [ ]:
!pip install onnxscript --quiet

model.eval()
dummy_input = torch.randn(1, 12, 1000).to(DEVICE)
torch.onnx.export(
    model,
    dummy_input,
    "ptbxl_cnn_deploy.onnx",
    input_names=["ecg_signal"],
    output_names=["diagnosis_logits"],
    dynamic_axes={"ecg_signal": {0: "batch_size"}, "diagnosis_logits": {0: "batch_size"}},
    opset_version=18 # Changed from 13 to 18 to match the exporter's preferred version
)
print("\nExported lightweight model to ptbxl_cnn_deploy.onnx for deployment.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 16.7 MB/s eta 0:00:00


/tmp/ipykernel_739/1799518593.py:5: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ECGCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ECGCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

Exported lightweight model to ptbxl_cnn_deploy.onnx for deployment.


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [5]:
import json
import numpy as np
import torch
import torch.nn as nn
import os
import sys
import zipfile
import pandas as pd
import wfdb
from scipy.signal import butter, filtfilt
from sklearn.metrics import roc_curve
from torchsummary import summary
from torch.utils.data import Dataset, DataLoader

In [ ]:
DATA_PATH = "/content/sample_data/data"
MODEL_SAVE_PATH = "/content/sample_data/ptbxl_cnn_best.pt"
ZIP_PATH = os.path.join(DATA_PATH, "records100.zip")
CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
SAMPLING_RATE = 100
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def ensure_extracted():
    """Ensure records100.zip is extracted and .hea files are present."""
    extracted_records_dir = os.path.join(DATA_PATH, "records100")
    zip_file_path = ZIP_PATH # ZIP_PATH is defined in K-eU2iPzUaX9

    # Check if the extracted directory exists and contains any .hea files
    hea_files_exist = False
    if os.path.exists(extracted_records_dir):
        # Perform a quick check for at least one .hea file
        for root, _, files in os.walk(extracted_records_dir):
            if any(f.endswith('.hea') for f in files):
                hea_files_exist = True
                break
        if hea_files_exist:
            print(f"Records already extracted in '{extracted_records_dir}'. No re-extraction needed.")
            return

    # If records don't exist or no .hea files found, proceed with extraction
    print(f"Records not found in '{extracted_records_dir}' or missing .hea files. Attempting extraction from '{zip_file_path}'...")
    if os.path.exists(zip_file_path):
        try:
            with zipfile.ZipFile(zip_file_path, 'r') as zf:
                zf.extractall(DATA_PATH) # Extracts to DATA_PATH/records100
            print("Extraction complete.")
        except zipfile.BadZipFile:
            print(f"Error: The zip file '{zip_file_path}' is corrupted.")
        except Exception as e:
            print(f"Error during extraction: {e}")
    else:
        print(f"Error: Zip file not found at '{zip_file_path}'. Cannot extract data.")

In [ ]:
ensure_extracted() # Ensure data is extracted before loading model.

model = ECGCNN().to(DEVICE)
model_full_path = MODEL_SAVE_PATH # MODEL_SAVE_PATH is already full path from K-eU2iPzUaX9
if os.path.exists(model_full_path):
    model.load_state_dict(torch.load(model_full_path, map_location=DEVICE))
    model.eval()
    print("✅ Model loaded from", model_full_path)
else:
    # This should ideally not happen if training was successful and path is correct.
    raise FileNotFoundError(f"Model file {model_full_path} not found. Train first or adjust path.")

Records not found in '/content/sample_data/data/records100' or missing .hea files. Attempting extraction from '/content/sample_data/data/records100.zip'...
Extraction complete.
✅ Model loaded from /content/sample_data/ptbxl_cnn_best.pt


In [ ]:
print("val_loader not found in session. Loading data from disk to compute thresholds...")
import pandas as pd
import wfdb
from scipy.signal import butter, filtfilt
from torch.utils.data import Dataset, DataLoader

val_loader not found in session. Loading data from disk to compute thresholds...


In [ ]:
    def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=100, order=4):
        nyq = 0.5 * fs
        b, a = butter(order, [lowcut/nyq, highcut/nyq], btype="band")
        return filtfilt(b, a, signal, axis=0)

    def preprocess_signal(signal):
        if signal.shape[0] == 12 and signal.shape[1] != 12:
            signal = signal.T
        signal = bandpass_filter(signal, fs=SAMPLING_RATE)
        mean = signal.mean(axis=0, keepdims=True)
        std = signal.std(axis=0, keepdims=True) + 1e-8
        signal = (signal - mean) / std
        return signal.astype(np.float32)

    class PTBXLDataset(Dataset):
        def __init__(self, dataframe, data_path, sampling_rate):
            self.df = dataframe
            self.data_path = data_path
            self.sampling_rate = sampling_rate

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            filepath = row["filename_lr"] if self.sampling_rate == 100 else row["filename_hr"]
            # Fixed path: The zip already contains 'records100', so we join data_path directly with filepath
            full_path = os.path.join(self.data_path, filepath)
            record = wfdb.rdrecord(full_path)
            signal = preprocess_signal(record.p_signal)
            return torch.tensor(signal.T, dtype=torch.float32), torch.tensor(row["label_vec"], dtype=torch.float32)

In [ ]:
import ast
df = pd.read_csv(os.path.join(DATA_PATH, "ptbxl_database.csv"), index_col="ecg_id")
df.scp_codes = df.scp_codes.apply(ast.literal_eval)
agg_df = pd.read_csv(os.path.join(DATA_PATH, "scp_statements.csv"), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = [agg_df.loc[k].diagnostic_class for k in y_dic.keys() if k in agg_df.index]
    return list(set(tmp))

df["diagnostic_superclass" ] = df.scp_codes.apply(aggregate_diagnostic)
df = df[df["diagnostic_superclass"].apply(len) > 0].copy()

def multi_hot(label_list):
    return np.array([1.0 if c in label_list else 0.0 for c in CLASSES], dtype=np.float32)
df["label_vec"] = df["diagnostic_superclass"].apply(multi_hot)

# Re-initialize with the corrected path logic from PTBXLDataset
val_df = df[df.strat_fold == 9]
val_ds = PTBXLDataset(val_df, DATA_PATH, SAMPLING_RATE)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("✅ val_loader re-initialized with fixed file paths.")

✅ val_loader re-initialized with fixed file paths.


In [ ]:
def find_optimal_thresholds(loader, model, device):
    model.eval()
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for signals, labels in loader:
            # Final safety check for shape (Batch, Channels, Length)
            if signals.shape[2] == 12 and signals.shape[1] != 12:
                signals = signals.transpose(1, 2)

            signals = signals.to(device)
            logits = model(signals)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    thresholds = {}
    for i, cls in enumerate(CLASSES):
        fpr, tpr, thresh = roc_curve(all_labels[:, i], all_probs[:, i])
        j = tpr - fpr
        best_idx = np.argmax(j)
        best_thresh = thresh[best_idx] if best_idx < len(thresh) else 0.5
        thresholds[cls] = float(best_thresh)
    return thresholds

optimal_thresholds = find_optimal_thresholds(val_loader, model, DEVICE)
print("Optimal thresholds:", optimal_thresholds)

Optimal thresholds: {'NORM': 0.3232671618461609, 'MI': 0.5393802523612976, 'STTC': 0.5180799961090088, 'CD': 0.4472930431365967, 'HYP': 0.5002087950706482}


In [ ]:
config = {
    "model_name": "ECGCNN",
    "model_version": "1.0",
    "input_shape": [12, 1000],
    "sampling_rate": SAMPLING_RATE,
    "num_leads": 12,
    "time_points": 1000,
    "classes": CLASSES,
    "class_names": {
        "NORM": "Normal",
        "MI": "Myocardial Infarction",
        "STTC": "ST/T Changes",
        "CD": "Conduction Disturbance",
        "HYP": "Hypertrophy"
    },
    "class_descriptions": {
        "NORM": "Normal ECG. No significant abnormalities detected.",
        "MI": "Signs of myocardial infarction (heart attack). Seek immediate medical attention.",
        "STTC": "ST/T wave changes indicating possible ischemia or other abnormalities.",
        "CD": "Conduction disturbance affecting the heart's electrical signaling.",
        "HYP": "Evidence of ventricular hypertrophy (enlarged heart muscle)."
    },
    "preprocessing": {
        "bandpass_lowcut": 0.5,
        "bandpass_highcut": 40.0,
        "bandpass_order": 4,
        "normalization": "per_record_zscore"
    },
    "model_parameters": {
        "in_channels": 12,
        "n_classes": 5,
        "conv_kernel_size": 7,
        "conv_padding": 3,
        "pool_size": 2,
        "fc_units": [256, 128, 5],
        "dropout": 0.4
    },
    "thresholds": optimal_thresholds,
    "training_config": {
        "batch_size": BATCH_SIZE,
        "epochs": 30,
        "learning_rate": 1e-3,
        "optimizer": "Adam",
        "loss_function": "BCEWithLogitsLoss"
    },
    "device": str(DEVICE)
}

# Save JSON
with open("model_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("✅ model_config.json saved")

✅ model_config.json saved


In [ ]:
preprocessing_code = '''
import numpy as np
from scipy.signal import butter, filtfilt

def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=100, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype="band")
    return filtfilt(b, a, signal, axis=0)

def preprocess_signal(signal):
    if signal.shape[0] == 12 and signal.shape[1] != 12:
        signal = signal.T
    signal = bandpass_filter(signal, fs=100)
    mean = signal.mean(axis=0, keepdims=True)
    std = signal.std(axis=0, keepdims=True) + 1e-8
    signal = (signal - mean) / std
    return signal.T.astype(np.float32)
'''
with open("preprocessing.py", "w") as f:
    f.write(preprocessing_code)
print("✅ preprocessing.py saved")

# Save model architecture summary
with open("model_architecture.txt", "w") as f:
    old_stdout = sys.stdout
    sys.stdout = f
    summary(model, (12, 1000))
    sys.stdout = old_stdout
print("✅ model_architecture.txt saved")

✅ preprocessing.py saved
✅ model_architecture.txt saved


In [ ]:
torch.save(model.state_dict(), "ptbxl_cnn_best.pt")
print("✅ Model weights backed up to ptbxl_cnn_best.pt")

print("\n🎉 All metadata files generated successfully!")

✅ Model weights backed up to ptbxl_cnn_best.pt

🎉 All metadata files generated successfully!


In [6]:
import torch
import torch.nn as nn
import numpy as np
import os
import wfdb
from scipy.signal import butter, filtfilt

In [16]:
CONFIG_PATH = "/content/model_config.json"
MODEL_PATH = "/content/ptbxl_cnn_best.pt"
DATA_PATH = "/content/sample_data/data"

In [8]:
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

CLASSES = config["classes"]
SAMPLING_RATE = config["sampling_rate"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [9]:
class ECGCNN(nn.Module):
    def __init__(self, in_channels=12, n_classes=5):
        super().__init__()
        self.block1 = self._conv_block(in_channels, 32)
        self.block2 = self._conv_block(32, 64)
        self.block3 = self._conv_block(64, 128)
        self.block4 = self._conv_block(128, 256)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, n_classes)
        )
    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv1d(in_c, out_c, kernel_size=7, padding=3),
            nn.BatchNorm1d(out_c), nn.ReLU(), nn.MaxPool1d(2)
        )
    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = self.gap(x).squeeze(-1)
        return self.fc(x)

In [10]:
model = ECGCNN(
    in_channels=config["model_parameters"]["in_channels"],
    n_classes=config["model_parameters"]["n_classes"]
).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("✅ Model loaded successfully from", MODEL_PATH)

✅ Model loaded successfully from /content/ptbxl_cnn_best.pt


In [11]:
def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=100, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype="band")
    return filtfilt(b, a, signal, axis=0)

def preprocess_signal(signal):
    """
    signal: numpy array of shape (leads, time) or (time, leads)
    Returns: (leads, time) normalized float32.
    """
    if signal.shape[0] == 12 and signal.shape[1] != 12:
        signal = signal.T
    signal = bandpass_filter(signal, fs=SAMPLING_RATE)
    mean = signal.mean(axis=0, keepdims=True)
    std = signal.std(axis=0, keepdims=True) + 1e-8
    signal = (signal - mean) / std
    return signal.T.astype(np.float32)

In [13]:
def predict(signal):
    """
    Perform inference on a raw ECG signal.

    Args:
        signal (np.ndarray): shape (12, 1000) or (1000, 12)

    Returns:
        dict: {
            'probabilities': {class_name: float},
            'predicted_class': str,
            'confidence': float
        }
    """
    # Preprocess
    processed = preprocess_signal(signal)        # (12, 1000)
    tensor = torch.tensor(processed).unsqueeze(0).to(DEVICE)  # (1,12,1000)

    # Inference
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()

    # Build result
    prob_dict = {cls: float(prob) for cls, prob in zip(CLASSES, probs)}
    pred_idx = np.argmax(probs)
    pred_class = CLASSES[pred_idx]
    confidence = float(probs[pred_idx])

    return {
        'probabilities': prob_dict,
        'predicted_class': pred_class,
        'confidence': confidence
    }

In [14]:
def apply_thresholds(prob_dict, thresholds=None):
    """
    Apply class‑specific thresholds to get binary predictions.
    If thresholds are not provided, uses default 0.5.
    """
    if thresholds is None:
        thresholds = config.get("thresholds", {cls: 0.5 for cls in CLASSES})
    result = {}
    for cls, prob in prob_dict.items():
        result[cls] = prob >= thresholds.get(cls, 0.5)
    return result


In [21]:
# ------------------- 8. Demo: Predict on a test record (with fallback) -------------------
if __name__ == "__main__":
    # Try to load a real test record from PTB-XL
    demo_signal = None
    if os.path.exists(os.path.join(DATA_PATH, "ptbxl_database.csv")):
        import pandas as pd
        df = pd.read_csv(os.path.join(DATA_PATH, "ptbxl_database.csv"), index_col="ecg_id")
        test_df = df[df.strat_fold == 10]

        # Find a record that actually exists
        for idx in range(len(test_df)):
            row = test_df.iloc[idx]
            filepath = row["filename_lr"] if SAMPLING_RATE == 100 else row["filename_hr"]
            full_path = os.path.join(DATA_PATH, filepath)
            # wfdb expects the .hea file; check existence
            if os.path.exists(full_path + ".hea"):
                try:
                    record = wfdb.rdrecord(full_path)
                    demo_signal = record.p_signal.T   # (12, 1000)
                    print(f"✅ Loaded demo record: {filepath}")
                    break
                except Exception as e:
                    print(f"⚠️ Failed to load {filepath}: {e}")
                    continue
        if demo_signal is None:
            print("⚠️ No valid test record found. Using a dummy signal for demo.")

    # If no real signal, create a synthetic one
    if demo_signal is None:
        # Generate a random 12-lead ECG (not clinically meaningful, just for demo)
        demo_signal = np.random.randn(12, 1000) * 0.5
        print("🔹 Using synthetic random signal (12 leads × 1000 samples).")

    # Run prediction
    result = predict(demo_signal)
    print("\n🔬 Demo prediction:")
    for cls, prob in result['probabilities'].items():
        print(f"  {cls}: {prob:.4f}")
    print(f"\n✅ Predicted class: {result['predicted_class']} (confidence: {result['confidence']:.4f})")

    # Optionally apply thresholds
    binary = apply_thresholds(result['probabilities'])
    print("\n🔲 Binary decisions (>= threshold):")
    for cls, val in binary.items():
        print(f"  {cls}: {'Abnormal' if val else 'Normal'}")

⚠️ No valid test record found. Using a dummy signal for demo.
🔹 Using synthetic random signal (12 leads × 1000 samples).

🔬 Demo prediction:
  NORM: 0.0047
  MI: 0.7652
  STTC: 0.8817
  CD: 0.8154
  HYP: 0.6561

✅ Predicted class: STTC (confidence: 0.8817)

🔲 Binary decisions (>= threshold):
  NORM: Normal
  MI: Abnormal
  STTC: Abnormal
  CD: Abnormal
  HYP: Abnormal


In [24]:
if __name__ == "__main__":
    # Check if PTB-XL data is available for a demo
    if os.path.exists(os.path.join(DATA_PATH, "ptbxl_database.csv")):
        import pandas as pd

        # Load full dataset for searching
        df = pd.read_csv(os.path.join(DATA_PATH, "ptbxl_database.csv"), index_col="ecg_id")
        # Original test_df is only used for context, we will iterate the full df to find an existing record
        # test_df = df[df.strat_fold == 10]

        found_record = False
        record = None # Initialize record to None
        raw_signal = None # Initialize raw_signal to None
        selected_row = None

        print("Searching for an available ECG record for the demo...")
        # Iterate through the entire DataFrame to find an existing record
        for idx in range(len(df)):
            row = df.iloc[idx]
            filepath = row["filename_lr"] if SAMPLING_RATE == 100 else row["filename_hr"]
            full_record_path = os.path.join(DATA_PATH, filepath)

            # Check if the .hea file exists before attempting to read
            if os.path.exists(full_record_path + '.hea'):
                # print(f"Attempting to load record: {full_record_path}")
                try:
                    record = wfdb.rdrecord(full_record_path)
                    raw_signal = record.p_signal.T   # (12, 1000)
                    found_record = True
                    selected_row = row # Keep track of the row for the record found
                    print(f"✅ Found and loaded record: {full_record_path}")
                    break # Found a valid record, break the loop
                except Exception as e:
                    print(f"Error reading record {full_record_path}: {e}. Trying next record.")
                    continue
            # else:
                # print(f"Record .hea file not found at: {full_record_path}.hea. Trying next record.")

        if found_record:
            # Predict using the found record
            result = predict(raw_signal)

            print("\n🔬 Demo prediction on an available record:")
            for cls, prob in result['probabilities'].items():
                print(f"  {cls}: {prob:.4f}")
            print(f"\n✅ Predicted class: {result['predicted_class']} (confidence: {result['confidence']:.4f})")

            # Show binary decision using thresholds
            binary = apply_thresholds(result['probabilities'])
            print("\n🔲 Binary decisions (>= threshold):")
            for cls, val in binary.items():
                print(f"  {cls}: {'Abnormal' if val else 'Normal'}")
        else:
            print("❌ No valid ECG records found on disk that could be loaded by wfdb from the entire dataset.")
            print("Please ensure the PTB-XL dataset (especially 'records100.zip') is correctly downloaded and extracted.")
    else:
        print("PTB-XL data (ptbxl_database.csv) not found. To run the demo, provide a signal manually.")
        print("\nExample usage on your own signal:")
        print("  my_signal = np.random.randn(12, 1000)   # replace with your data")
        print("  result = predict(my_signal)")
        print("  print(result)")

Searching for an available ECG record for the demo...
✅ Found and loaded record: /content/sample_data/data/records100/00000/00001_lr

🔬 Demo prediction on an available record:
  NORM: 0.9663
  MI: 0.0256
  STTC: 0.0302
  CD: 0.0109
  HYP: 0.0160

✅ Predicted class: NORM (confidence: 0.9663)

🔲 Binary decisions (>= threshold):
  NORM: Abnormal
  MI: Normal
  STTC: Normal
  CD: Normal
  HYP: Normal
